### RAG Pipeline- Data ingestion to vector DB pipeline


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path 

In [5]:
### Read all the PDFs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all the PDF files in a directory"""
    
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")

        try:
            # Load the PDF file using PyPDFLoader
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

# Process all PDfs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process
Processing ..\data\pdf\Resume.pdf
Loaded 1 pages

Total documents loaded: 1


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-22T17:25:03+00:00', 'author': '', 'keywords': '', 'moddate': '2026-09-22T17:25:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume.pdf', 'file_type': 'pdf'}, page_content='Anmol Yadav\nUttar Pradesh, India\n♂phone+91-9336044988 —✉yadavanmol540@gmail.com —/gl⌢bePortfolio —/linkedinLinkedIn —/githubGitHub\nEDUCATION\nKanpur Institute of TechnologySep 2023 – Jun 2027\nBachelor of Technology (Computer Science and Engineering) –CGPA: 8.19 (Up to 6th Semester)Kanpur, Uttar Pradesh\nTECHNICAL SKILLS\nLanguages:C, C++, Python, JavaScript (ES6+), TypeScript, HTML5, CSS3, SQL\nFrontend & Mobile:React.js, Next.js, React Native, Expo, Tailwind CSS\nBackend & APIs:Node

In [7]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split into {len(documents)} documents into {len(split_docs)} chunks")

    #show example pf a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split into 1 documents into 5 chunks

Example chunk:
Content:Anmol Yadav
Uttar Pradesh, India
♂phone+91-9336044988 —✉yadavanmol540@gmail.com —/gl⌢bePortfolio —/linkedinLinkedIn —/githubGitHub
EDUCATION
Kanpur Institute of TechnologySep 2023 – Jun 2027
Bachelor ...
Metadata: {'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-22T17:25:03+00:00', 'author': '', 'keywords': '', 'moddate': '2026-09-22T17:25:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-22T17:25:03+00:00', 'author': '', 'keywords': '', 'moddate': '2026-09-22T17:25:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume.pdf', 'file_type': 'pdf'}, page_content='Anmol Yadav\nUttar Pradesh, India\n♂phone+91-9336044988 —✉yadavanmol540@gmail.com —/gl⌢bePortfolio —/linkedinLinkedIn —/githubGitHub\nEDUCATION\nKanpur Institute of TechnologySep 2023 – Jun 2027\nBachelor of Technology (Computer Science and Engineering) –CGPA: 8.19 (Up to 6th Semester)Kanpur, Uttar Pradesh\nTECHNICAL SKILLS\nLanguages:C, C++, Python, JavaScript (ES6+), TypeScript, HTML5, CSS3, SQL\nFrontend & Mobile:React.js, Next.js, React Native, Expo, Tailwind CSS\nBackend & APIs:Node

### embedding and vectorstoreDB

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\anmol\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class EmbeddingManager:
    """Handles documents embedding generation using sentence-transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Model name: Hugging Face sentence-transformer model."""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence-transformers model."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. "
                f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, documents: List[str]) -> np.ndarray:
        """Generate embeddings for a list of documents."""
        if self.model is None:
            raise ValueError("Model is not loaded.")

        print(f"Generating embeddings for {len(documents)} documents")

        embeddings = self.model.encode(
            documents,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape {embeddings.shape}")

        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the dimension of the embeddings."""
        if self.model is None:
            raise ValueError("Model is not loaded.")

        return self.model.get_sentence_embedding_dimension()


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

# Check embedding dimension
embedding_manager.get_embedding_dimension()

Loading model: all-MiniLM-L6-v2


c:\Users\anmol\Desktop\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anmol\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\anmol\Desktop\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarn

Model loaded successfully. Embedding dimension: 384


C:\Users\anmol\AppData\Local\Temp\ipykernel_16772\4178971423.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
C:\Users\anmol\AppData\Local\Temp\ipykernel_16772\4178971423.py:44: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


384